# 🤟 Wasel v4 Pro — Live Sign Language Translator
### YOLO Skeleton + MediaPipe Hands + Pre-Trained Classifier

**Architecture:**
- 🔍 **YOLO** → Person detection + body skeleton (fast, GPU)
- 🖐️ **MediaPipe** → Hand & finger keypoints (225 features)
- 🧠 **Classifier** → Pre-trained PSL model → live translation

**Steps:** Cell 1 (Install) → Restart Runtime → Cell 2 (Launch!)

> ⚡ **Hybrid engine: YOLO speed + MediaPipe accuracy = best of both!**

In [ ]:
#@title 📦 Cell 1: Install (then Restart Runtime → run Cell 2)

!pip install -q "gradio>=5.0.0" "mediapipe==0.10.14" "ultralytics" "scikit-learn>=1.3.0" "opencv-python-headless"

import mediapipe as mp
print(f"mediapipe {mp.__version__} — solutions: {'✅' if hasattr(mp, 'solutions') else '❌'}")
from ultralytics import YOLO
print(f"ultralytics ✅")
print("\n⚠️ Now: Runtime → Restart session → then run Cell 2 (skip this cell!)")

In [ ]:
#@title 🎥 Cell 2: Launch Live Translator
#@markdown YOLO skeleton + MediaPipe hands + pre-trained model!

import os, cv2, pickle, logging
import numpy as np
import mediapipe as mp
import gradio as gr
from ultralytics import YOLO
from collections import deque

logging.basicConfig(level=logging.INFO)
print(f"mediapipe: {mp.__version__}")

# ═══════════════════════════════════════
# 1. Clone repo & load pre-trained model
# ═══════════════════════════════════════
REPO = "/content/wasel_repo"
if not os.path.exists(REPO):
    print("⏳ Cloning repository...")
    !git clone -q https://github.com/eltaweelactuary/Wasel_v4_Official_Release.git {REPO}

PKL = os.path.join(REPO, "GCP_Source_Code/wasel_v4_data/models/psl_classifier.pkl")
with open(PKL, 'rb') as f:
    classifier, label_encoder = pickle.load(f)
N_FEAT = classifier.n_features_in_
print(f"✅ Classifier: {N_FEAT} features, signs={list(label_encoder.classes_)}")

# ═══════════════════════════════════════
# 2. Load YOLO Pose (body skeleton)
# ═══════════════════════════════════════
yolo = YOLO("yolov8n-pose.pt")
print("✅ YOLO Pose loaded (GPU)" if next(yolo.model.parameters()).is_cuda else "✅ YOLO Pose loaded (CPU)")

# ═══════════════════════════════════════
# 3. Load MediaPipe (hand features)
# ═══════════════════════════════════════
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils
mp_styles = mp.solutions.drawing_styles
holistic = mp_holistic.Holistic(
    static_image_mode=False, model_complexity=0,
    min_detection_confidence=0.5, min_tracking_confidence=0.5
)
print("✅ MediaPipe Holistic loaded (hands + features)")

# ═══════════════════════════════════════
# 4. Feature extraction (225 features)
# ═══════════════════════════════════════
def extract_features(results):
    features = []
    if results.pose_landmarks:
        for lm in results.pose_landmarks.landmark:
            features.extend([lm.x, lm.y, lm.z])
    else:
        features.extend([0.0] * 99)
    if results.left_hand_landmarks:
        for lm in results.left_hand_landmarks.landmark:
            features.extend([lm.x, lm.y, lm.z])
    else:
        features.extend([0.0] * 63)
    if results.right_hand_landmarks:
        for lm in results.right_hand_landmarks.landmark:
            features.extend([lm.x, lm.y, lm.z])
    else:
        features.extend([0.0] * 63)
    feat = np.array(features)
    if len(feat) > N_FEAT: feat = feat[:N_FEAT]
    elif len(feat) < N_FEAT: feat = np.pad(feat, (0, N_FEAT - len(feat)))
    return feat

# YOLO skeleton drawing config
YOLO_SKELETON = [
    (0,1),(0,2),(1,3),(2,4),  # face
    (5,6),(5,7),(7,9),(6,8),(8,10),  # arms
    (5,11),(6,12),(11,12),  # torso
    (11,13),(13,15),(12,14),(14,16)  # legs
]

def draw_yolo_skeleton(frame, keypoints, conf_kps):
    """Draw YOLO pose skeleton with thick colored bones."""
    h, w = frame.shape[:2]
    pts = []
    for i in range(17):
        x, y, c = float(keypoints[i][0]), float(keypoints[i][1]), float(conf_kps[i]) if conf_kps is not None else 1.0
        if c > 0.3:
            px, py = int(x), int(y)
            pts.append((px, py))
            cv2.circle(frame, (px, py), 5, (0, 255, 255), -1)  # yellow dots
        else:
            pts.append(None)
    # Draw bones
    for (a, b) in YOLO_SKELETON:
        if a < len(pts) and b < len(pts) and pts[a] and pts[b]:
            cv2.line(frame, pts[a], pts[b], (0, 200, 255), 3)  # orange bones
    return frame

# ═══════════════════════════════════════
# 5. Frame processing pipeline
# ═══════════════════════════════════════
buf = deque(maxlen=10)

def translate(image):
    if image is None:
        return image
    try:
        # ── YOLO: detect person + draw body skeleton ──
        yolo_results = yolo(image, verbose=False, conf=0.3)
        out = image.copy()
        for r in yolo_results:
            if r.keypoints is not None and len(r.keypoints.xy) > 0:
                kps = r.keypoints.xy[0].cpu().numpy()
                conf_kps = r.keypoints.conf[0].cpu().numpy() if r.keypoints.conf is not None else None
                out = draw_yolo_skeleton(out, kps, conf_kps)

        # ── MediaPipe: extract hand features (silent) ──
        rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mp_results = holistic.process(rgb)

        # Draw hand skeletons on top of YOLO body
        if mp_results.left_hand_landmarks:
            mp_drawing.draw_landmarks(out, mp_results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS, mp_styles.get_default_hand_landmarks_style())
        if mp_results.right_hand_landmarks:
            mp_drawing.draw_landmarks(out, mp_results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS, mp_styles.get_default_hand_landmarks_style())

        # ── Classify sign ──
        feat = extract_features(mp_results)
        buf.append(feat)
        text = "Waiting for signs..."
        if len(buf) >= 3:
            avg = np.mean(np.array(list(buf)), axis=0).reshape(1, -1)
            probs = classifier.predict_proba(avg)[0]
            idx = np.argmax(probs)
            conf = float(probs[idx]) * 100
            if conf > 30.0:
                sign = label_encoder.inverse_transform([idx])[0]
                text = f"{sign.upper()} ({conf:.1f}%)"

        # Draw translation text
        cv2.rectangle(out, (10, 10), (600, 65), (0, 0, 0), -1)
        cv2.putText(out, text, (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.3, (0, 255, 0), 3)
        return out
    except Exception as e:
        logging.error(f"Error: {e}")
        return image

# ═══════════════════════════════════════
# 6. Launch!
# ═══════════════════════════════════════
demo = gr.Interface(
    fn=translate,
    inputs=gr.Image(sources=["webcam"], streaming=True, label="📸 Camera"),
    outputs=gr.Image(label="🤟 Live Translation"),
    live=True,
    title="🤟 Wasel v4 Pro — Live Sign Language Translator",
    description="YOLO detects your body + MediaPipe tracks your fingers = AI translates your signs!",
)
print("\n🎉 Launching Wasel v4 Live!")
print("📋 Share the public URL with anyone!\n")
demo.launch(share=True, quiet=False)